# SAS to Databricks: Linear and Polynomial Regression

This beginner notebook shows how a simple SAS regression workflow maps to Databricks and PySpark.

You will learn how to:
- Create a small training dataset
- Convert input columns into a PySpark `features` vector
- Train a linear regression model
- Add polynomial features
- Compare model results with common regression metrics


## 1. Import Libraries

In Databricks, `spark` is already available in the notebook session. The imports below add the PySpark ML classes used in this lesson.


In [ ]:
import warnings

import numpy as np
import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, VectorAssembler
from pyspark.ml.regression import LinearRegression
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Libraries are ready")
print(f"Spark version: {spark.version}")


## 2. Create Example Data

SAS users often start with a `DATA` step. Here we create the same kind of table with Python, then convert it into a Spark DataFrame.


In [ ]:
np.random.seed(42)
row_count = 100

house_data = pd.DataFrame({
    "square_footage": np.random.uniform(1000, 5000, row_count),
    "bedrooms": np.random.randint(1, 5, row_count),
    "bathrooms": np.random.randint(1, 4, row_count),
    "age": np.random.randint(0, 50, row_count),
})

house_data["price"] = (
    150 * house_data["square_footage"]
    + 50000 * house_data["bedrooms"]
    - 2000 * house_data["age"]
    + np.random.normal(0, 20000, row_count)
)

houses = spark.createDataFrame(house_data)

print(f"Rows: {houses.count()}")
houses.show(5)
houses.describe().show()


## 3. SAS Reference: Create Example Data

In SAS, a `DATA` step creates rows and columns. The Python/PySpark version creates a pandas DataFrame first, then converts it to a Spark DataFrame.

```sas
DATA houses;
  DO row_id = 1 TO 100;
    square_footage = 1000 + RAND('UNIFORM') * 4000;
    bedrooms = CEIL(RAND('UNIFORM') * 4);
    age = FLOOR(RAND('UNIFORM') * 50);
    price = 150 * square_footage + 50000 * bedrooms - 2000 * age + RAND('NORMAL') * 20000;
    OUTPUT;
  END;
RUN;

PROC PRINT DATA=houses (OBS=5);
RUN;
```


## 4. Visualize the Example Data with Matplotlib

Matplotlib is the main Python library for basic charts. We import it as `plt` because that is the common short name used in notebooks.

In this chart, each dot is one house. The x-axis shows square footage and the y-axis shows price. This helps you see whether a regression line is a reasonable model.


In [ ]:
plot_data = houses.select("square_footage", "price").toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(plot_data["square_footage"], plot_data["price"], alpha=0.7)
plt.title("House Price by Square Footage")
plt.xlabel("Square Footage")
plt.ylabel("Price")
plt.grid(True)
plt.show()


## 5. Prepare Features

SAS lets you write model inputs directly in the `MODEL` statement. PySpark ML expects one vector column named `features`.


In [ ]:
feature_columns = ["square_footage"]
target_column = "price"

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
model_data = assembler.transform(houses).withColumnRenamed(target_column, "label")

model_data.select("square_footage", "features", "label").show(5, truncate=False)


## 6. Train a Linear Regression Model

This is the PySpark version of a simple `PROC REG` model.


In [ ]:
linear_regression = LinearRegression(featuresCol="features", labelCol="label")
linear_model = linear_regression.fit(model_data)

linear_predictions = linear_model.transform(model_data)

print("Linear regression model")
print(f"Intercept: {linear_model.intercept:,.2f}")
print(f"square_footage coefficient: {linear_model.coefficients[0]:,.2f}")

linear_predictions.select("square_footage", "label", "prediction").show(5)


## 7. SAS Reference: Linear Regression

SAS can train a simple linear regression with `PROC REG`. In PySpark, the same idea uses `VectorAssembler` plus `LinearRegression`.

```sas
PROC REG DATA=houses;
  MODEL price = square_footage;
  OUTPUT OUT=linear_predictions P=predicted_price;
RUN;
QUIT;
```


## 8. Train a Polynomial Regression Model

Polynomial regression adds curved terms such as `square_footage^2`. This can help when the relationship is not perfectly straight.


In [ ]:
polynomial_expansion = PolynomialExpansion(
    degree=2,
    inputCol="features",
    outputCol="polynomial_features",
)

polynomial_data = polynomial_expansion.transform(model_data)

polynomial_regression = LinearRegression(
    featuresCol="polynomial_features",
    labelCol="label",
)
polynomial_model = polynomial_regression.fit(polynomial_data)
polynomial_predictions = polynomial_model.transform(polynomial_data)

print("Polynomial regression model")
print(f"Intercept: {polynomial_model.intercept:,.2f}")
print(f"square_footage coefficient: {polynomial_model.coefficients[0]:,.4f}")
print(f"square_footage squared coefficient: {polynomial_model.coefficients[1]:,.6f}")

polynomial_predictions.select("square_footage", "label", "prediction").show(5)


## 9. SAS Reference: Polynomial Regression

Polynomial regression adds curved terms. In SAS, you can create the squared column first, then include it in the model.

```sas
DATA houses_poly;
  SET houses;
  square_footage_sq = square_footage * square_footage;
RUN;

PROC REG DATA=houses_poly;
  MODEL price = square_footage square_footage_sq;
  OUTPUT OUT=poly_predictions P=predicted_price;
RUN;
QUIT;
```


## 10. Visualize Linear and Polynomial Predictions

This chart overlays the real prices with the model predictions. The closer the prediction lines are to the dots, the better the model is fitting this example.


In [ ]:
linear_plot = linear_predictions.select("square_footage", "label", "prediction").toPandas()
polynomial_plot = polynomial_predictions.select("square_footage", "label", "prediction").toPandas()

linear_plot = linear_plot.sort_values("square_footage")
polynomial_plot = polynomial_plot.sort_values("square_footage")

plt.figure(figsize=(9, 5))
plt.scatter(polynomial_plot["square_footage"], polynomial_plot["label"], alpha=0.45, label="Actual")
plt.plot(linear_plot["square_footage"], linear_plot["prediction"], label="Linear prediction")
plt.plot(polynomial_plot["square_footage"], polynomial_plot["prediction"], label="Polynomial prediction")
plt.title("Actual Prices vs Model Predictions")
plt.xlabel("Square Footage")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()


## 11. Compare the Models

RMSE and MAE are error values, so lower is better. R2 measures fit quality, so higher is better.


In [ ]:
rmse_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

linear_rmse = rmse_evaluator.evaluate(linear_predictions)
linear_mae = mae_evaluator.evaluate(linear_predictions)
linear_r2 = r2_evaluator.evaluate(linear_predictions)

polynomial_rmse = rmse_evaluator.evaluate(polynomial_predictions)
polynomial_mae = mae_evaluator.evaluate(polynomial_predictions)
polynomial_r2 = r2_evaluator.evaluate(polynomial_predictions)

print(f"{'Metric':<10} {'Linear':>15} {'Polynomial':>15}")
print("-" * 42)
print(f"{'RMSE':<10} {linear_rmse:>15,.0f} {polynomial_rmse:>15,.0f}")
print(f"{'MAE':<10} {linear_mae:>15,.0f} {polynomial_mae:>15,.0f}")
print(f"{'R2':<10} {linear_r2:>15.4f} {polynomial_r2:>15.4f}")


## 12. Visualize Model Error

RMSE and MAE are error metrics, so smaller bars are better.


In [ ]:
metric_names = ["RMSE", "MAE"]
linear_scores = [linear_rmse, linear_mae]
polynomial_scores = [polynomial_rmse, polynomial_mae]
x_positions = np.arange(len(metric_names))
bar_width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x_positions - bar_width / 2, linear_scores, bar_width, label="Linear")
plt.bar(x_positions + bar_width / 2, polynomial_scores, bar_width, label="Polynomial")
plt.title("Linear vs Polynomial Error")
plt.xticks(x_positions, metric_names)
plt.ylabel("Error")
plt.legend()
plt.grid(axis="y")
plt.show()


## 13. SAS to Databricks Mapping

| Task | SAS | Databricks |
| --- | --- | --- |
| Create data | `DATA ... RUN;` | Spark DataFrame |
| Explore data | `PROC PRINT`, `PROC MEANS` | `show()`, `describe()` |
| Select model inputs | `MODEL price = x;` | `VectorAssembler` |
| Train regression | `PROC REG` | `LinearRegression` |
| Add curved terms | `PROC GLM` interactions | `PolynomialExpansion` |
| Score data | `OUTPUT OUT=` | `model.transform()` |

Start with a linear model first. Add polynomial features only when the data or business problem suggests a curved relationship.
